In [2]:
import random


CHROM_LENGTH = 5


def binary_to_int(chromosome):
    return int(chromosome, 2)


def fitness(chromosome):
    x = binary_to_int(chromosome)
    return x ** 2


def get_int_input(prompt, default):
    raw = input(f"{prompt} [default = {default}]: ").strip()
    if raw == "":
        return default
    try:
        return int(raw)
    except ValueError:
        return default


def get_chromosome_input(index):
    while True:
        raw = input(f"Enter chromosome {index} (5-bit binary, e.g. 11011): ").strip()
        if len(raw) == CHROM_LENGTH and all(bit in '01' for bit in raw):
            return raw
        print(f"Invalid input. Enter exactly {CHROM_LENGTH} bits of 0/1.")


def get_population_input(pop_size):
    print(f"Enter {pop_size} chromosomes for the initial population.")
    return [get_chromosome_input(i + 1) for i in range(pop_size)]


def random_chromosome():
    return ''.join(random.choice('01') for _ in range(CHROM_LENGTH))


def create_random_population(pop_size):
    return [random_chromosome() for _ in range(pop_size)]


def evaluate_population(population):
    fitnesses = [fitness(c) for c in population]
    total_fitness = sum(fitnesses)
    avg_fitness = total_fitness / len(population)
    probabilities = [f / total_fitness if total_fitness > 0 else 0 for f in fitnesses]
    expected_counts = [f / avg_fitness if avg_fitness > 0 else 0 for f in fitnesses]
    return fitnesses, probabilities, expected_counts, total_fitness, avg_fitness


def print_table(population, fitnesses, probabilities, expected_counts):
    print(f"{'No':<4}{'Chromosome':<14}{'x':<6}{'fitness':<10}{'prob':<10}{'expected_count':<15}")
    for i, chromo in enumerate(population):
        x = binary_to_int(chromo)
        print(f"{i + 1:<4}{chromo:<14}{x:<6}{fitnesses[i]:<10}{probabilities[i]:<10.3f}{expected_counts[i]:<15.3f}")


def roulette_wheel_selection(population, probabilities):
    pick = random.random()
    cumulative = 0.0
    for chromo, prob in zip(population, probabilities):
        cumulative += prob
        if pick <= cumulative:
            return chromo
    return population[-1]


def crossover(parent1, parent2):
    point = random.randint(1, CHROM_LENGTH - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2


def mutate(chromosome, mutation_rate):
    genes = list(chromosome)
    for i in range(len(genes)):
        if random.random() < mutation_rate:
            genes[i] = '1' if genes[i] == '0' else '0'
    return ''.join(genes)


def next_generation(population, probabilities, mutation_rate):
    new_population = []
    while len(new_population) < len(population):
        parent1 = roulette_wheel_selection(population, probabilities)
        parent2 = roulette_wheel_selection(population, probabilities)
        child1, child2 = crossover(parent1, parent2)
        child1 = mutate(child1, mutation_rate)
        child2 = mutate(child2, mutation_rate)
        new_population.append(child1)
        if len(new_population) < len(population):
            new_population.append(child2)
    return new_population


def main():
    pop_size = get_int_input("population_size", 4)
    generations = get_int_input("iterations", 5)
    mutation_rate = float(get_int_input("mutation_rate_percent", 5)) / 100

    mode = input("random or user-input population? [r/u, default r]: ").strip().lower()
    if mode == "u":
        population = get_population_input(pop_size)
    else:
        random.seed()
        population = create_random_population(pop_size)

    best_ever = max(population, key=fitness)

    for gen in range(generations):
        fitnesses, probabilities, expected_counts, total_fitness, avg_fitness = evaluate_population(population)

        print(f"\n--- Generation {gen + 1} ---")
        print_table(population, fitnesses, probabilities, expected_counts)
        print(f"total_fitness = {total_fitness}")
        print(f"average_fitness = {avg_fitness:.3f}")

        current_best = max(population, key=fitness)
        if fitness(current_best) > fitness(best_ever):
            best_ever = current_best

        population = next_generation(population, probabilities, mutation_rate)

    print(f"\nbest_chromosome = {best_ever}")
    print(f"best_x = {binary_to_int(best_ever)}")
    print(f"best_fitness = {fitness(best_ever)}")


if __name__ == "__main__":
    main()

population_size [default = 4]: 4
iterations [default = 5]: 20
mutation_rate_percent [default = 5]: 5
random or user-input population? [r/u, default r]: u
Enter 4 chromosomes for the initial population.
Enter chromosome 1 (5-bit binary, e.g. 11011): 11011
Enter chromosome 2 (5-bit binary, e.g. 11011): 00011
Enter chromosome 3 (5-bit binary, e.g. 11011): 00101
Enter chromosome 4 (5-bit binary, e.g. 11011): 10001

--- Generation 1 ---
No  Chromosome    x     fitness   prob      expected_count 
1   11011         27    729       0.693     2.772          
2   00011         3     9         0.009     0.034          
3   00101         5     25        0.024     0.095          
4   10001         17    289       0.275     1.099          
total_fitness = 1052
average_fitness = 263.000

--- Generation 2 ---
No  Chromosome    x     fitness   prob      expected_count 
1   11011         27    729       0.255     1.019          
2   11011         27    729       0.255     1.019          
3   11010      